# PySpark Repartition - optimization



# Initalise a spark session

In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("RepartitionExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


26/05/20 23:47:21 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/20 23:47:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-56b80e51-b13c-4f39-88d8-c91be67580e4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 181ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

# Repartition in PySpark

repartition() in PySpark
repartition() is used to increase or decrease the number of partitions in a DataFrame/RDD. It performs a full shuffle of the data across the cluster.

# The basic syntax

df.repartition(numPartitions, *cols)


# 1. Repartition by Number

In [2]:

df = spark.range(100)

print("Before:", df.rdd.getNumPartitions())  # e.g., 4

df_repartitioned = df.repartition(10)

print("After:", df_repartitioned.rdd.getNumPartitions())  # 10

Before: 16
After: 10


# 2. Repartition by Column


In [5]:
data = [("Alice", "HR"), ("Bob", "IT"), ("Carol", "HR"), ("Dave", "IT"), ("Eve", "Finance")]
df = spark.createDataFrame(data, ["name", "dept"])
df.show()
# Each unique dept value gets its own partition(s)
df_by_dept = df.repartition("dept")
df_by_dept.show()

print("Partitions:", df_by_dept.rdd.getNumPartitions())


+-----+-------+
| name|   dept|
+-----+-------+
|Alice|     HR|
|  Bob|     IT|
|Carol|     HR|
| Dave|     IT|
|  Eve|Finance|
+-----+-------+

+-----+-------+
| name|   dept|
+-----+-------+
|Alice|     HR|
|  Bob|     IT|
|Carol|     HR|
| Dave|     IT|
|  Eve|Finance|
+-----+-------+

Partitions: 1


Why You Got 1 Partition
You're running in local mode with a small dataset. Here's why:

Root Cause
pythonspark = SparkSession.builder.appName("test").getOrCreate()
# No .master() specified → defaults to local[1] = 1 core = 1 partition

AQE automatically coalesces empty/tiny partitions into 1 — so instead of 200 partitions (197 empty + 3 with data), AQE merges them all down to 1.



# Verify AQE is the cause

In [8]:
# Disable AQE and try again
spark.conf.set("spark.sql.adaptive.enabled", "false")

df_by_dept = df.repartition("dept")
print(df_by_dept.rdd.getNumPartitions())  # Now shows 200

200


# Reset/Enable AQE

In [13]:
spark.conf.set("spark.sql.adaptive.enabled","true")  # "true" by default in Spark 3.x
spark.conf.get("spark.sql.adaptive.enabled")  # "true" by default in Spark 3.x


'true'

# 3. Repartition by Number + Column

In [7]:
# 5 partitions, distributed by dept column
df_by_dept_5 = df.repartition(5, "dept")

print("Partitions:", df_by_dept_5.rdd.getNumPartitions())

df_by_dept_5.show()


Partitions: 5
+-----+-------+
| name|   dept|
+-----+-------+
|  Bob|     IT|
| Dave|     IT|
|Alice|     HR|
|Carol|     HR|
|  Eve|Finance|
+-----+-------+



# Fix 2 — Drop Duplicate Columns After Join

In [5]:
joined = orders.join(customers, orders.customer_id == customers.id)

# Drop the customer 'id' and 'name' using the DataFrame reference
# (must use DataFrame.col — string "id" would be ambiguous)
result = joined.drop(customers["id"]).drop(customers["name"])

result.show()

+---+------+-----------+---------------+
| id|  name|customer_id|          email|
+---+------+-----------+---------------+
|  1|Laptop|        101|alice@email.com|
|  3|Tablet|        101|alice@email.com|
|  2| Phone|        102|  bob@email.com|
+---+------+-----------+---------------+



# Fix 3 — Join on Shared Key with USING-style (string key)

In [11]:
# When join key has the SAME name in both DataFrames,
# pass it as a string — PySpark keeps only one copy automatically
orders2 = orders.withColumnRenamed("customer_id", "id")  # rename to match

result = orders2.join(customers, on="id")   # single 'id' in output

result.show()

+---+----+---+----+-----+
| id|name| id|name|email|
+---+----+---+----+-----+
+---+----+---+----+-----+



# Fix 4 — Rename Before Joining (Cleanest for Pipelines)

In [12]:
orders_clean = (orders
    .withColumnRenamed("id",   "order_id")
    .withColumnRenamed("name", "order_name")
)

customers_clean = (customers
    .withColumnRenamed("id",   "customer_id")
    .withColumnRenamed("name", "customer_name")
)

result = orders_clean.join(customers_clean, on="customer_id")

result.show()

+-----------+--------+----------+-------------+---------------+
|customer_id|order_id|order_name|customer_name|          email|
+-----------+--------+----------+-------------+---------------+
|        101|       1|    Laptop|        Alice|alice@email.com|
|        101|       3|    Tablet|        Alice|alice@email.com|
|        102|       2|     Phone|          Bob|  bob@email.com|
+-----------+--------+----------+-------------+---------------+

